# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 3

In this version, we organize the training process of the neural character model.

The model keeps the same training text, character vocabulary, numerical representation, weight matrix, softmax probabilities and autoregressive generator.

The examples are divided into training and validation sets. The model learns from shuffled mini-batches, while training and validation loss are measured after every epoch.

This version introduces a structured and reproducible training process without changing the model architecture.

## 1. Imports and Configuration

The configuration collects the values that control the experiment.

Using one explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import numpy as np

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character transitions.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.
The neural model also assigns an integer identifier to every character so that characters can be represented numerically.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Character pairs and numerical encoding

Each character is converted into its integer identifier.

For the text `model`:

`model`  
↓  
`[15, 17, 7, 8, 14]`

Adjacent identifiers form the training examples:

- `m -> o` becomes `15 -> 17`
- `o -> d` becomes `17 -> 7`
- `d -> e` becomes `7  -> 8`
- `e -> l` becomes `8  -> 14`

The first identifier is the input. The following identifier is the target to predict.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14]`  
↓  
`model`

In [5]:
examples = list(zip(corpus, corpus[1:]))

input_ids = np.array([character_to_id[current_character] for current_character, _ in examples])
target_ids = np.array([character_to_id[next_character] for _, next_character in examples])

print("Number of examples:", len(examples))
print("First 12 examples:", examples[:12])
print("First 12 input IDs:", input_ids[:12])
print("First 12 target IDs:", target_ids[:12])

Number of examples: 439
First 12 examples: [('l', 'a'), ('a', 'n'), ('n', 'g'), ('g', 'u'), ('u', 'a'), ('a', 'g'), ('g', 'e'), ('e', ' '), (' ', 'm'), ('m', 'o'), ('o', 'd'), ('d', 'e')]
First 12 input IDs: [14  4 16 10 22  4 10  8  1 15 17  7]
First 12 target IDs: [ 4 16 10 22  4 10  8  1 15 17  7  8]


## 3. Neural Model

### One-hot inputs and trainable parameters

Each input character is represented by a one-hot vector. A one-hot vector contains one value equal to 1 and all other values equal to 0.

Multiplying this representation by the weight matrix produces one score for every possible next character. These scores are called logits.

In [6]:
inputs = np.eye(vocabulary_size)[input_ids]

model_random = np.random.default_rng(SEED)

initial_weights = model_random.normal(loc=0.0, scale=0.01, size=(vocabulary_size, vocabulary_size))

print("Input matrix shape:", inputs.shape)
print("Weight matrix shape:", initial_weights.shape)
print("Trainable parameters:", initial_weights.size)

Input matrix shape: (439, 27)
Weight matrix shape: (27, 27)
Trainable parameters: 729


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

The indices are shuffled with a local random generator, making the split reproducible.

In [7]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))

split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = inputs[train_indices]
train_targets = target_ids[train_indices]

validation_inputs = inputs[validation_indices]
validation_targets = target_ids[validation_indices]

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))

Training examples: 351
Validation examples: 88


### Softmax probabilities

The weight matrix produces logits, which are unrestricted numerical scores.

Softmax converts the logits into probabilities:

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

Subtracting the maximum logit before exponentiation improves numerical stability.

In [8]:
def softmax(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)

    exponentials = np.exp(shifted_logits)

    return exponentials / exponentials.sum(axis=1, keepdims=True)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

A high probability for the correct character produces a low loss. A low probability produces a high loss.

Training updates the weights to reduce this value.

In [9]:
def calculate_loss(weights, inputs, targets):
    logits = inputs @ weights
    probabilities = softmax(logits)

    correct_probabilities = probabilities[np.arange(len(targets)), targets]

    return -np.mean(np.log(correct_probabilities + 1e-12))

In [10]:
initial_train_loss = calculate_loss(initial_weights, train_inputs, train_targets)

initial_validation_loss = calculate_loss(initial_weights, validation_inputs, validation_targets)

print("Initial train loss:", initial_train_loss)
print("Initial validation loss:", initial_validation_loss)

Initial train loss: 3.2972156217892037
Initial validation loss: 3.297493118750668


### Training with mini-batch gradient descent

Training is organized into epochs.

During every epoch:

1. the training examples are shuffled;
2. the examples are divided into mini-batches;
3. the model calculates predictions for one mini-batch;
4. the gradients are calculated;
5. the weights are updated;
6. training and validation loss are measured.

The validation examples are never used to update the weights.

The gradients remain implemented directly with NumPy, without automatic differentiation.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its weights after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [11]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [12]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_weights = initial_weights.copy()

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = calculate_loss(trained_weights, train_inputs, train_targets)

        validation_loss = calculate_loss(trained_weights, validation_inputs, validation_targets)

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = train_inputs[shuffled_indices]
        shuffled_targets = train_targets[shuffled_indices]

        for batch_inputs, batch_targets in create_batches(shuffled_inputs, shuffled_targets, batch_size):
            logits = batch_inputs @ trained_weights
            probabilities = softmax(logits)

            logits_gradient = probabilities.copy()

            logits_gradient[np.arange(len(batch_targets)), batch_targets] -= 1

            logits_gradient /= len(batch_targets)

            weights_gradient = (batch_inputs.T @ logits_gradient)

            trained_weights -= (learning_rate * weights_gradient)

    return (trained_weights, train_loss_history, validation_loss_history)

In [13]:
(
    trained_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))

best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss)
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss)
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2972 | Validation loss: 3.2975
Epoch  10 | Train loss: 2.6810 | Validation loss: 3.0164
Epoch  20 | Train loss: 2.4053 | Validation loss: 2.9371
Epoch  30 | Train loss: 2.2641 | Validation loss: 2.9126
Epoch  40 | Train loss: 2.1790 | Validation loss: 2.9022
Epoch  50 | Train loss: 2.1217 | Validation loss: 2.8966
Epoch  60 | Train loss: 2.0803 | Validation loss: 2.8946
Epoch  70 | Train loss: 2.0488 | Validation loss: 2.8943
Epoch  80 | Train loss: 2.0240 | Validation loss: 2.8953
Epoch  90 | Train loss: 2.0040 | Validation loss: 2.8986
Epoch 100 | Train loss: 1.9876 | Validation loss: 2.9037

Initial train loss: 3.2972156217892037
Final train loss: 1.9876393217154849
Initial validation loss: 3.297493118750668
Final validation loss: 2.9036790222735935
Best validation loss: 2.8937998396623015
Best validation epoch: 64


### Reading the losses

Training loss measures the examples used to update the model.

Validation loss measures separate examples that do not update the weights.

In this run, the validation loss reaches its lowest value at epoch 64 and then increases slightly. This is possible because only the training loss is directly optimized.

The reported values help compare learning on the training examples with performance on separate validation examples.

### Learned probabilities

After training, the model can produce a next-character probability distribution for any character in the vocabulary.

The probabilities now come from the trained weights instead of direct transition counts.

The probabilities are learned from the training set. The validation set measures the model without updating its parameters.

In [14]:
def next_character_probabilities(weights, current_character):
    if current_character not in character_to_id:
        raise ValueError("The character is not in the vocabulary.")

    current_id = character_to_id[current_character]
    logits = weights[current_id:current_id + 1]
    probabilities = softmax(logits)[0]

    return {
        id_to_character[index]: probability
        for index, probability in enumerate(probabilities)
    }

In [15]:
probabilities_after_m = next_character_probabilities(trained_weights, "m")

most_likely_after_m = sorted(
    probabilities_after_m.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

print("Most likely characters after 'm':")

for character, probability in most_likely_after_m:
    print(f"{repr(character):>4}: {probability:.4f}")

print("Total:", sum(probabilities_after_m.values()))

Most likely characters after 'm':
 'o': 0.2497
 'e': 0.1818
 'a': 0.1810
 's': 0.1126
 'p': 0.1120
 ' ': 0.0427
 'u': 0.0058
 'l': 0.0058
 'i': 0.0058
 'h': 0.0058
Total: 0.9999999999999998


## 4. Generator

### Sample the next character

The current character selects one row of trained logits.

Softmax converts the logits into probabilities, and the next character is sampled from that distribution.

The sampled character becomes the new input for the following step.

In [16]:
def sample_next_character(weights, current_id, random_generator):
    logits = weights[current_id:current_id + 1]
    probabilities = softmax(logits)[0]

    return random_generator.choice(vocabulary_size, p=probabilities)

In [17]:
demo_random = np.random.default_rng(SEED)
m_id = character_to_id["m"]

sampled_ids = [sample_next_character(trained_weights, m_id, demo_random) for _ in range(5)]

print("Five samples after 'm':", [id_to_character[index] for index in sampled_ids])

Five samples after 'm': ['p', 'e', 's', 'o', 'a']


### Generate text

Generation repeats the same loop:

1. read the current character;
2. sample the next character;
3. append it to the output;
4. use the new character as the next context.

The seed makes the example reproducible.

The probabilities are produced by the trained parameters instead of being calculated directly from transition counts.

In [18]:
def generate_text(weights, start_character="i", length=200, seed=42):
    if length < 1:
        raise ValueError("Length must be at least 1.")

    if start_character not in character_to_id:
        raise ValueError("The start character is not in the vocabulary.")

    random_generator = np.random.default_rng(seed)

    generated = [start_character]
    current_id = character_to_id[start_character]

    for _ in range(length - 1):
        current_id = sample_next_character(weights, current_id, random_generator)

        generated.append(id_to_character[current_id])

    return "".join(generated)

In [19]:
generated_text = generate_text(trained_weights, start_character="i", length=300, seed=SEED)

print(generated_text)

iomsicvor mexts.
l tiomttr m erntecod maretndts, s f
tinmin.
s.
ting amams aimonmompa.
th marecheccextndysmg cth inctonerng m me codxter wio mounealee tt.
cs smpaud nets windy amxtousearne s we t.
ns iouna, dg moncton.
wimontex,er edylspks ctrangunecleundcts co.
e weace ad pomelpr ffg.
.
l t.
cttwor


## 5. Tests

These assertions verify the numerical representation, data split, model shapes, probability distributions, training behavior and reproducibility of training and generation.

The training procedure is repeated from the same initial weights and with the same seed to verify that it produces the same weights and loss histories.

In [20]:
assert len(examples) == len(corpus) - 1
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)
assert inputs.shape == (len(examples), vocabulary_size)
assert initial_weights.shape == (vocabulary_size, vocabulary_size)
assert trained_weights.shape == (vocabulary_size, vocabulary_size)
assert np.allclose(inputs.sum(axis=1), 1.0)
assert (len(train_inputs) + len(validation_inputs) == len(inputs))
assert len(train_inputs) == len(train_targets)
assert (len(validation_inputs) == len(validation_targets))
assert len(np.intersect1d(train_indices, validation_indices)) == 0
assert len(train_loss_history) == EPOCHS + 1
assert (len(validation_loss_history) == EPOCHS + 1)
assert final_train_loss < initial_train_loss
assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))
assert abs(sum(next_character_probabilities(trained_weights, "m").values()) - 1.0) < 1e-12

(
    repeated_weights,
    repeated_train_history,
    repeated_validation_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    print_every=None
)

assert np.allclose(trained_weights, repeated_weights)
assert np.allclose(train_loss_history, repeated_train_history)
assert np.allclose(validation_loss_history, repeated_validation_history)
assert generate_text(trained_weights, "l", 30, seed=10) == generate_text(trained_weights, "l", 30, seed=10)
assert len(generate_text(trained_weights, "l", 30, seed=10)) == 30

print("All checks passed.")

All checks passed.


## Notes

- The model architecture remains a character-level neural bigram model.
- The examples are divided into reproducible training and validation sets.
- Training examples are shuffled at the beginning of every epoch.
- Mini-batches replace full-batch parameter updates.
- Training loss measures performance on the examples used for learning.
- Validation loss measures performance on separate examples.
- The validation examples never update the model parameters.
- The same seed reproduces initialization, data splitting, training and generation.
- The generator remains autoregressive and samples one character at a time.
- The tests verify the data split, training behavior and reproducibility.

Future versions will introduce new components and gradually evolve the architecture.